# CP6 - Operaciones Avanzadas II

### Como siempre, antes de empezar...

Si has cerrado el proyecto o has abierto `Nueva Ventana` en VS Code, pulsa en el primer icono del menú de la izquierda `Explorador` y después en `Abrir Carpeta`. 

Busca la ruta en la que guardas el repositorio de GitHub en tu PC, normalmente *`Documentos => GitHub => fundamentos_big_data` * y pulsa en `Abrir`. De esta forma, ya tendrás la carpeta de tu repositorio cargada en VS Code.

> ¡Importante! Antes de abrir esa carpeta, debes ir a GitHub Desktop y realizar un `pull` para asegurarte de que tus archivos están actualizados. También puedes hacer el pull desde VS Code.  

### Requisitos para arrancar el entonro
- Iniciar Docker (abrir docker desktop).
- Iniciar GitHub (abrir github desktop) y estar logueado.
- En el terminal, ejecutar el siguiente comando en la raíz del proyecto `fundamentos_big_data %` 
```bash
docker-compose up --build
```
> ¡NO TE OLVIDES! Docker debe estar abierto y en ejecución en tu pc para poder utilizarlo. 


### Seleccionar Kernel
- Seleccionar kernel
- Servidor de Juypter existente
- Escriba la dirección URL del servidor de Jupyter en ejecución
- Seleccionar http://127.0.0.1:8888/?token=xxxxxxxx y pulsar Enter
- 127.0.0.1 y pulsar Enter
- Seleccionar Python 3 (ipykernel)

___________________

## Objetivos de la sesión

- Funciones para el manejo de strings.
- Funciones para trabajar con colecciones.
- Otras funciones y UDF.

In [ ]:
### Importar las librerías necesarias ###
### INICIA LA SPARK-SESSION ###
### Añade el spark context ###
### Añade el path para acceder a los datos y nómbralo como DATA_PATH###

## 2 - Dataframes en PySpark

### 2.13 Funciones para Strings

In [2]:
df_str = spark.createDataFrame([(' Spark ',)], ['nombre'])
df_str.show()

+-------+
| nombre|
+-------+
| Spark |
+-------+



> **Nota**: el valor 'Spark' de la columna "nombre", tiene un espacio en blanco a la izquierda y otro a la derecha.

#### **Funciones `ltrim()`, `rtrim()` y `trim()`**

Son el equivalente en python a `lstrip()`, `rstrip()` y `strip()`.

In [3]:
df_str.select(
    F.ltrim('nombre').alias('ltrim'),
    F.rtrim('nombre').alias('rtrim'),
    F.trim('nombre').alias('trim')
).show()

+------+------+-----+
| ltrim| rtrim| trim|
+------+------+-----+
|Spark | Spark|Spark|
+------+------+-----+



#### **Funciones `lpad()` y `rpad()`**

Sirve para rellenar a la izquierda o a la derecha con caracteres alfanuméricos. Se le pasa como parámetros:

- Columna que queremos seleccionar.
- Número total de caracteres que tendrá tras la transformación.
- Caracter de relleno.

In [4]:
df_str.select(
    F.trim('nombre').alias('trim')
).select(
    F.lpad(F.col('trim'), 8, '-').alias('lpad'),
    F.rpad(F.col('trim'), 8, '=').alias('rpad')
).show()

+--------+--------+
|    lpad|    rpad|
+--------+--------+
|---Spark|Spark===|
+--------+--------+



#### **Funciones `concat_ws()`, `lower()`, `upper()`, `initcap()` y `reverse()`**

La función `concat_ws` tiene dos parámetros:

- Caracter de separación.
- Columna a la que aplicarlo.

In [5]:
df_str2 = spark.createDataFrame([('Spark','es','maravilloso')],
                                ['sujeto', 'verbo', 'adjetivo'])
df_str2.show()

+------+-----+-----------+
|sujeto|verbo|   adjetivo|
+------+-----+-----------+
| Spark|   es|maravilloso|
+------+-----+-----------+



In [6]:
df_str2.select(
    F.concat_ws(' ', 
        F.col('sujeto'),
        F.col('verbo'),
        F.col('adjetivo')).alias('frase'),
).select(
    F.col('frase'),
    F.lower(F.col('frase')).alias('minuscula'),
    F.upper(F.col('frase')).alias('mayuscula'),
    F.initcap(F.col('frase')).alias('capital'),
    F.reverse(F.col('frase')).alias('reverse')
).show()

+--------------------+--------------------+--------------------+--------------------+--------------------+
|               frase|           minuscula|           mayuscula|             capital|             reverse|
+--------------------+--------------------+--------------------+--------------------+--------------------+
|Spark es maravilloso|spark es maravilloso|SPARK ES MARAVILLOSO|Spark Es Maravilloso|osollivaram se krapS|
+--------------------+--------------------+--------------------+--------------------+--------------------+



#### **Función `regexp_replace()`**

La función tiene 3 parámetros principales:

- Columna a la que aplicaremos cambios.
- Término(s) que queremos sustituir.
- Término que se añadirá.

In [7]:
df_str3 = spark.createDataFrame([('voy a casa a por mis llaves',)], ['frase'])

In [8]:
df_str3.select(
    F.regexp_replace(F.col('frase'), 'voy|por', 'ir').alias('nueva_frase')
).show(truncate=False)

+-------------------------+
|nueva_frase              |
+-------------------------+
|ir a casa a ir mis llaves|
+-------------------------+



### 2.14 Funciones para trabajar con Colecciones

In [9]:
df_c = spark.read.parquet(DATA_PATH + 'colecciones')

In [10]:
df_c.show(truncate=False)

+-----+--------------------------------------------+
|dia  |tareas                                      |
+-----+--------------------------------------------+
|lunes|[hacer la tarea, buscar agua, lavar el auto]|
+-----+--------------------------------------------+



In [11]:
df_c.printSchema()

root
 |-- dia: string (nullable = true)
 |-- tareas: array (nullable = true)
 |    |-- element: string (containsNull = true)



En este caso, tenemos un Schema que nos muestra que una de las variables es de tipo `array` y que incluye una colección de elementos de tipo `str`.

#### **Función `size()`, `sort_array()` y `array_contains()`**

In [12]:
df_c.select(
    F.size(F.col('tareas')).alias('size'),
    F.sort_array(F.col('tareas')).alias('array_ordenado'),
    F.array_contains(F.col('tareas'), 'buscar agua').alias('contiene...')
).show(truncate=False)

+----+--------------------------------------------+-----------+
|size|array_ordenado                              |contiene...|
+----+--------------------------------------------+-----------+
|3   |[buscar agua, hacer la tarea, lavar el auto]|true       |
+----+--------------------------------------------+-----------+



#### **Función `explode()`**

Nos permite "deconstruir" un array en filas de una columna seleccionada.

In [13]:
df_c.select(
    F.col('dia'),
    F.explode(F.col('tareas')).alias('tareas')
).show()

+-----+--------------+
|  dia|        tareas|
+-----+--------------+
|lunes|hacer la tarea|
|lunes|   buscar agua|
|lunes| lavar el auto|
+-----+--------------+



### 2.15 Otras funciones

#### **Función `when()`**

La función `when()` sirve para establecer condiciones. Se le introducen estos parámetros:

- Columna con condición lógica.
- Salida de esa condición.

Se puede añadir un valor por defecto usando la función `otherwise()`.

In [14]:
df_p = spark.createDataFrame([('Jose',1),('Julia',2),('Katia',1),
                              (None, 3),('Raul',3)], ['nombre','pago'])
df_p.show()

+------+----+
|nombre|pago|
+------+----+
|  Jose|   1|
| Julia|   2|
| Katia|   1|
|  NULL|   3|
|  Raul|   3|
+------+----+



In [15]:
df_p.select(
    F.col('nombre'),
    (F.when(F.col('pago') == 1, 'pagado')
    .when(F.col('pago') == 2, 'no pagado')
    .otherwise('sin iniciar')).alias('status')
).show()

+------+-----------+
|nombre|     status|
+------+-----------+
|  Jose|     pagado|
| Julia|  no pagado|
| Katia|     pagado|
|  NULL|sin iniciar|
|  Raul|sin iniciar|
+------+-----------+



#### **Funciones `coalesce()` y `lit()`**

La función `coalesce()` va a recibir `n` columnas y va a devolver el **primer valor que no sea nulo**. Por su parte, `lit()` nos permite añadir un literal. En el ejemplo, como tenemos un valor a `null`, lo vamos a detectar con `coalesce()` y luego renombrarlo usando `lit()`.

In [16]:
df_p.select(
    F.coalesce(F.col('nombre'), F.lit('sin nombre')).alias('nombre')
).show()

+----------+
|    nombre|
+----------+
|      Jose|
|     Julia|
|     Katia|
|sin nombre|
|      Raul|
+----------+



## 3 - Funciones definidas por el usuario (UDF)

**FORMA 1**: Vamos a ver la primera forma de registrar funciones `udf` en spark.

In [17]:
def cubo(n):
  return n**3

Para registrar una función `udf`, tenemos que usar la estructura `spark.udf.register()`. Los parámetros que le pasaremos serán:

- Nombre de la udf.
- Nombre de la función previamente definida.
- Tipo de dato que va a devolver.

In [18]:
spark.udf.register('cubo', cubo, T.LongType())

<function __main__.cubo(n)>

In [19]:
df_u = spark.range(1,10) # creamos un df numérico con range()

In [20]:
df_u.show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



Una vez que tenemos registrada nuestra función, ya la podemos integrar. En este caso, le pasamos la columna con los enteros del df creado anteriormente.

In [21]:
df_u.select(
    F.col('id'),
    cubo(F.col('id')).alias('cubo')
).show()

+---+-----+
| id| cubo|
+---+-----+
|  1|  1.0|
|  2|  8.0|
|  3| 27.0|
|  4| 64.0|
|  5|125.0|
|  6|216.0|
|  7|343.0|
|  8|512.0|
|  9|729.0|
+---+-----+



**FORMA 2**: Vamos a ver la segunda forma de crear funciones `udf`.

In [22]:
def bienvenida(nombre):
    return ('Hola, {}'.format(nombre))

In [23]:
bienvenida_udf = F.udf(lambda x: bienvenida(x), T.StringType())

In [24]:
df_u2 = spark.createDataFrame([('Jose',),('Julia',),('Katia',)], ['nombre'])

In [25]:
df_u2.show()

+------+
|nombre|
+------+
|  Jose|
| Julia|
| Katia|
+------+



In [26]:
df_u2.select(
    F.col('nombre'),
    bienvenida_udf(F.col('nombre')).alias('Saludo')
).show(truncate=False)

+------+-----------+
|nombre|Saludo     |
+------+-----------+
|Jose  |Hola, Jose |
|Julia |Hola, Julia|
|Katia |Hola, Katia|
+------+-----------+



**FORMA 3**: Vamos a ver la tercera forma de crear funciones `udf`. Se trata de usar un decorador.

In [28]:
@F.udf(returnType=T.StringType())

def mayuscula(s):
  return s.upper()

In [29]:
df_u2.select(
    F.col('nombre'),
    mayuscula(F.col('nombre')).alias('mayuscula')
).show(truncate=False)

+------+---------+
|nombre|mayuscula|
+------+---------+
|Jose  |JOSE     |
|Julia |JULIA    |
|Katia |KATIA    |
+------+---------+



## 4 - Funciones de Ventana

In [30]:
df_w = spark.read.parquet(DATA_PATH + 'ventanas')

In [31]:
df_w.show()

+------+------------+------+
|nombre|departamento|puntos|
+------+------------+------+
|Lazaro|      letras|    45|
|  Raul|  matemática|    42|
| Maria|  matemática|    31|
|  Jose|     química|    33|
|Susana|     química|    34|
|  Juan|      letras|    37|
| Julia|      letras|    30|
| Kadir|arquitectura|    21|
|Lilian|arquitectura|    42|
+------+------------+------+



Para crear una ventana (`Window`) tenemos que especificarle por qué columna vamos a particionar los datos y en función de qué columna vamos a ordenar la partición.

In [ ]:
### Departamento es el por la que asignamos las ventanas
### Tener la ventana ordenada por puntos
windowSpec = Window.partitionBy('departamento').orderBy('puntos')

Una vez hemos creado la ventana, podemos crear un sistema de ordenación por número de fila para cada partición. Para ello usamos `withColumn` y `row_number`, además del método `over()` para pasarle la ventana.

In [33]:
df_w.withColumn('row_number', F.row_number()\
    .over(windowSpec)).show(truncate=False)

+------+------------+------+----------+
|nombre|departamento|puntos|row_number|
+------+------------+------+----------+
|Kadir |arquitectura|21    |1         |
|Lilian|arquitectura|42    |2         |
|Julia |letras      |30    |1         |
|Juan  |letras      |37    |2         |
|Lazaro|letras      |45    |3         |
|Maria |matemática  |31    |1         |
|Raul  |matemática  |42    |2         |
|Jose  |química     |33    |1         |
|Susana|química     |34    |2         |
+------+------------+------+----------+



**Agregaciones con funciones de Ventana**

In [36]:
winSpecAgg = Window.partitionBy('departamento')

In [ ]:
(df_w.withColumn('row_number', F.row_number().over(windowSpec)) # ventana ordenada
    .withColumn('min', F.min(F.col('puntos')).over(winSpecAgg)) # ventana agregada
    .withColumn('max', F.max(F.col('puntos')).over(winSpecAgg)) 
    .withColumn('avg', F.avg(F.col('puntos')).over(winSpecAgg))
).show()

+------+------------+------+----------+---+---+------------------+
|nombre|departamento|puntos|row_number|min|max|               avg|
+------+------------+------+----------+---+---+------------------+
| Kadir|arquitectura|    21|         1| 21| 42|              31.5|
|Lilian|arquitectura|    42|         2| 21| 42|              31.5|
| Julia|      letras|    30|         1| 30| 45|37.333333333333336|
|  Juan|      letras|    37|         2| 30| 45|37.333333333333336|
|Lazaro|      letras|    45|         3| 30| 45|37.333333333333336|
| Maria|  matemática|    31|         1| 31| 42|              36.5|
|  Raul|  matemática|    42|         2| 31| 42|              36.5|
|  Jose|     química|    33|         1| 33| 34|              33.5|
|Susana|     química|    34|         2| 33| 34|              33.5|
+------+------------+------+----------+---+---+------------------+



Como aparecen elementos duplicados, es necesario añadir una condición `where` para filtrar por `row_number`.

In [37]:
(df_w.withColumn('row_number', F.row_number().over(windowSpec)) # ventana ordenada
    .withColumn('min', F.min(F.col('puntos')).over(winSpecAgg)) # ventana agregada
    .withColumn('max', F.max(F.col('puntos')).over(winSpecAgg))
    .withColumn('avg', F.avg(F.col('puntos')).over(winSpecAgg))
).where(F.col('row_number') == 1).select(
    F.col('departamento'), F.col('min'), F.col('max'), F.col('avg')
).show()

+------------+---+---+------------------+
|departamento|min|max|               avg|
+------------+---+---+------------------+
|arquitectura| 21| 42|              31.5|
|      letras| 30| 45|37.333333333333336|
|  matemática| 31| 42|              36.5|
|     química| 33| 34|              33.5|
+------------+---+---+------------------+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 46422)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

## Ejercicios

Vamos a utilizar los datos de vuelos. Lo primero que debes hacer es leer el dataset como `df` o `df_v`

#### **Ejercicio 1 - Retraso medio y desviación por aerolínea y mes**

Para vuelos no cancelados, calcula `avg_departure_delay` por `AIRLINE` y `MONTH`. Ordena por mayor retraso medio.

**Resultado esperado (formato + ejemplo):**
```
+--------+-----+--------------------+---------+
|AIRLINE |MONTH|avg_departure_delay |n_flights|
+--------+-----+--------------------+---------+
| UA     |  12 | 18.7               | 12034   |
| DL     |   7 | 16.1               |  9540   |
```

#### **Ejercicio 2 - Tasa de puntualidad por aeropuerto de origen**

Para cada `ORIGIN_AIRPORT`, calcula:

* `is_ontime` => vuelos con `ARRIVAL_DELAY <= 0`
* `pct_on_time` => % de vuelos con `ARRIVAL_DELAY <= 0` (y `CANCELLED='0'`)
* `avg_arr_delay`

Muestra top 10 aeropuertos con mayor `pct_on_time`.

**Importante**: tienes que usar Window para seleccionar `ORIGIN_AIRPORT`.

**Resultado esperado (formato + ejemplo):**
```
+---------------+-----------+--------------+----------+
|ORIGIN_AIRPORT |pct_on_time|avg_arr_delay |  count   |
+---------------+-----------+--------------+----------+
| BOI           | 87.2      | -1.3         |   1523   |
| SLC           | 86.5      |  0.8         |   3891   |
...
```

#### **Ejercicio 3 - Rutas más concurridas y con mayor % de cancelación**

Agrupa por (`ORIGIN_AIRPORT`, `DESTINATION_AIRPORT`), calcula `num_flights`, `%cancelled` y ordena por `num_flights` descendente. Devuelve top 15.

Además, crea un segundo dataset con top 15, ordenado por `%cancelled` (con al menos 200 vuelos).

**Resultado esperado (formato + ejemplo):**

```
top_busy:
+--------------+-------------------+-----------+-------------+
|ORIGIN_AIRPORT|DESTINATION_AIRPORT|num_flights|pct_cancelled|
+--------------+-------------------+-----------+-------------+
| ATL          | MCO               |   14560   |    1.9      |
| LAX          | SFO               |   13840   |    2.3      |
...

top_cancel:
+--------------+-------------------+-----------+-------------+
|ORD           |EWR                |    2200   |    5.7      |
|JFK           |BOS                |    1850   |    5.2      |
...
```

#### **Ejercicio 4: Enriquecer con tabla de aerolíneas y top distancia media**

En base al siguiente *lookup* de nombres de aerolínea (simulado), haz `join` por `AIRLINE`. Calcula la **distancia media** por aerolínea (no cancelados) y ordénalo desc.

```python
airlines_lookup = spark.createDataFrame(
    [("AA","American Airlines"),("DL","Delta Air Lines"),("UA","United Airlines")],
    ["AIRLINE","AIRLINE_NAME"]
)
```

**Resultado esperado (formato + ejemplo):**

```
+-------+-------------------+------------+------+
|AIRLINE|AIRLINE_NAME       |avg_distance|n     |
+-------+-------------------+------------+------+
| UA    | United Airlines   | 1243.5     | 18234|
| AA    | American Airlines | 1198.2     | 21567|
| DL    | Delta Air Lines   | 1104.7     | 20780|
```

---

#### **Ejercicio 5 - Top 5 destinos por aerolínea (lista ordenada)**

Para cada `AIRLINE`, construye una **lista ordenada** con los 5 destinos más frecuentes en `DESTINATION_AIRPORT`. Usa `collect_list` + `sort_array` con `struct`.

**Resultado esperado (formato + ejemplo):**

```
+-------+---------------------------+
|AIRLINE|top5_destinations          |
+-------+---------------------------+
|AA     |[DFW, ORD, MIA, LAX, CLT] |
|DL     |[ATL, DTW, MSP, LGA, JFK] |
...
```

#### **Ejercicio 6 - Clasificar vuelos por distancia y *pivot* por mes**

Define un UDF `distance_bucket(DISTANCE)` que devuelva `short` (<500), `medium` (500–1499) o `long` (≥1500). Calcula el **reparto** por `MONTH` y bucket (pivot), con recuentos.

**Resultado esperado (formato + ejemplo):**

```
+-----+-----+------+------+
|MONTH|short|medium| long |
+-----+-----+------+------+
|  1  | 4200| 8300 | 2100 |
|  2  | 3980| 7900 | 1985 |
...
```

#### **Ejercicio Extra - Causas de retraso: mayor causa por vuelo**

Crea array de pares `(delay_minutes, causa)` con `AIR_SYSTEM_DELAY, SECURITY_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, WEATHER_DELAY`. Obtén la **causa con más minutos** por vuelo.

**Resultado esperado (formato + ejemplo):**

```
+-------------+-------------+---------------+--------------+
+FLIGHT_NUMBER+ARRIVAL_DELAY+TOP_DELAY_CAUSE+TOP_DELAY_MINS+
+-------------+-------------+---------------+--------------+
|  1234       |   85        | LATE_AIRCRAFT | 70           |
|   987       |   42        | WEATHER       | 40           |
...
```